# Capstone: the cafe quarterly pack

Python stage, lesson P25. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** stack monthly files with `concat`, reconcile each step with `assert`, reshape the quarter into one table with one chart, and write a note whose numbers all come from variables.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# The cafes send one orders file a month. The export is simulated here: this cell cuts orders.csv into six files in a folder called monthly.
from pathlib import Path

Path("monthly").mkdir(exist_ok=True)
month_of = pd.to_datetime(orders["ordered_at"]).dt.month
for m in range(1, 7):
    orders[month_of == m].to_csv(f"monthly/orders_2026_{m:02d}.csv", index=False)
# Two lookups, as in lessons P13 and P15: a price for each product and a name for each store.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
names = dict(zip(stores["store_id"], stores["store_name"]))

## Unit 1: Stack the monthly files with concat

**Predict first.** the three files hold 349, 380 and 353 orders. What shape will `stack` have, and will its row labels be unique?

In [ ]:
frames = [pd.read_csv(f"monthly/orders_2026_{m:02d}.csv") for m in (4, 5, 6)]
stack = pd.concat(frames)
print(stack.shape, stack.index.is_unique)

**Predict first.** will the labels be unique now, and what will the largest label be?

In [ ]:
stack = pd.concat(frames, ignore_index=True)
print(stack.index.is_unique, stack.index.max())

## Unit 2: Reconcile each step with assert

**Predict first.** will `lines` have fewer, the same or more rows than the 3,287 lines of the whole file? Will any assert stop the cell? What will it print?

In [ ]:
lines = order_items.merge(stack[["order_id", "store_id", "ordered_at"]], on="order_id", validate="many_to_one")
lines = lines.assign(amount=lines["quantity"] * lines["product_id"].map(price) - lines["discount_usd"])
# the second route: pick the quarter's lines with isin, price them with map, never merge
picked = order_items[order_items["order_id"].isin(stack["order_id"])]
picked_total = (picked["quantity"] * picked["product_id"].map(price) - picked["discount_usd"]).sum()
assert stack["status"].eq("completed").all(), "the pack counts completed orders only"
assert len(lines) == len(picked), "the merge changed the number of lines"
assert round(lines["amount"].sum(), 2) == round(picked_total, 2), "the two totals differ"
print(len(lines), round(lines["amount"].sum(), 2))

**Predict first.** `stack` has 1,082 rows. What will the cell print?

In [ ]:
show_error('assert len(stack) == 1800, "the quarter has fewer orders than the whole file"', short=True)

## Unit 3: Replay: reshape the quarter and chart it

**Predict first.** which store sold the most in April, which in May and which in June?

In [ ]:
lines = lines.assign(store=lines["store_id"].map(names), month=pd.to_datetime(lines["ordered_at"]).dt.month)
monthly = lines.pivot_table(index="month", columns="store", values="amount", aggfunc="sum")
print(monthly.round(2))

**Predict first.** which store has the tallest bar in April, and where does the y axis start?

In [ ]:
fig, ax = plt.subplots()
monthly.plot.bar(ax=ax, rot=0)
ax.set_title("Sales by store and month, April to June 2026")
ax.set_xlabel("Month of 2026")
ax.set_ylabel("Sales (USD)")
ax.legend(title="store", loc="upper left", bbox_to_anchor=(1, 1))  # the legend beside the bars, so it hides none
print(monthly.loc[4].idxmax(), ax.get_ylim()[0])

## Unit 4: Write the note from variables

**Predict first.** does the order of the three stores change when the 20 counts as 2? Which store leads, and by how many dollars before and after?

In [ ]:
typo = lines["order_item_id"] == 6952
order_no, said, meant = lines.loc[typo, "order_id"].iloc[0], lines.loc[typo, "quantity"].iloc[0], 2
fixed = np.where(typo, meant * lines["product_id"].map(price) - lines["discount_usd"], lines["amount"])
totals = lines.groupby("store")["amount"].sum().sort_values(ascending=False)
fixed_totals = lines.assign(amount=fixed).groupby("store")["amount"].sum().sort_values(ascending=False)
first, second = totals.index[0], totals.index[1]
lead, lead_fixed = totals.iloc[0] - totals.iloc[1], fixed_totals.iloc[0] - fixed_totals.iloc[1]
note = (f"{first} sold the most from April to June 2026: {totals.iloc[0]:,.2f} dollars as recorded, {lead:,.2f} ahead of {second}. "
        f"One line of order {order_no} says {said} units where {meant} were meant; counted as {meant}, {first} leads by {lead_fixed:,.2f}. "
        f"Harbor, open since April 1, sold {totals['Harbor']:,.2f} in its first quarter.")
print(list(fixed_totals.index) == list(totals.index))
print(note)

## Finish it

### Exercise 1: Finish it

**Task:** How many different customers placed an order at Old Town from January to March? `q1_frames` holds the three monthly files for January, February and March. Stack them with the row labels numbered again, keep the orders of Old Town, store 1, and count the customers who placed an order. A walk-in order has no `customer_id`: it is not a customer.

**Expected:** `answer_1` is a whole number.

In [ ]:
q1_frames = [pd.read_csv(f"monthly/orders_2026_{m:02d}.csv") for m in (1, 2, 3)]  # given: the three files
# stack them, keep store 1, count the different customers, and set answer_1
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 67, traps={68: "`unique()` lists a missing id as one more value. Which method counts only the real ids?"})

<details>
<summary>Hint</summary>

Lesson P11 counted different values of a column with a method that leaves out missing values.

</details>

<details>
<summary>Solution</summary>

```python
q1_frames = [pd.read_csv(f"monthly/orders_2026_{m:02d}.csv") for m in (1, 2, 3)]
q1 = pd.concat(q1_frames, ignore_index=True)
answer_1 = q1[q1["store_id"] == 1]["customer_id"].nunique()
```

67 different customers. `nunique()` counts the ids that exist; `len(...unique())` counts the missing id of the walk-in order, which was taken at Old Town, as one more, and gives 68, which the check cell reports.

</details>

## On your own

### Exercise 2: On your own

**Task:** How much did each loyalty tier spend from April to June? Before you start, say in one sentence what the result looks like: three tiers, one dollar amount each. `lines` holds the quarter's order lines with their amounts, `stack` the quarter's orders with their `customer_id`, and `customers` the loyalty tier of each customer. Put the dollars in `answer_2` as a list in the alphabetical order of the tiers (bronze, gold, silver), each rounded to 2 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of three amounts.

In [ ]:
_ = check("Exercise 2", "answer_2", [8096.0, 1216.75, 4661.25], traps={(1136, 188, 655): "These numbers count lines. Which column holds the amounts in dollars?"})

<details>
<summary>Hint</summary>

`lines` has no `customer_id`. Bring it over from `stack` with a merge, then bring the tier over from `customers`, each with `validate`. Group by the tier and sum the amounts.

</details>

<details>
<summary>Solution</summary>

```python
with_tier = lines.merge(stack[["order_id", "customer_id"]], on="order_id", validate="many_to_one")
with_tier = with_tier.merge(customers[["customer_id", "loyalty_tier"]], on="customer_id", validate="many_to_one")
answer_2 = with_tier.groupby("loyalty_tier")["amount"].sum().round(2).tolist()
```

Bronze 8,096.00, gold 1,216.75 and silver 4,661.25: 13,974.00 together, the total of unit 2. Counting rows with `size()` gives 1,136, 188 and 655 lines, which are counts, not dollars, and the check cell asks which column holds the amounts.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. The February file came from a till whose export names the customer column `customer`. A colleague stacked the three files for January to March and did not look at the shape. Fix the lines so that the stack has the six columns of the other files, then set `answer_3` to the shape of the stack and the number of orders without a customer id.

**Expected:** `answer_3` is `[(718, 6), 1]`.

In [ ]:
jan = pd.read_csv("monthly/orders_2026_01.csv")
feb = pd.read_csv("monthly/orders_2026_02.csv").rename(columns={"customer_id": "customer"})  # the February file: another header
mar = pd.read_csv("monthly/orders_2026_03.csv")
stack_q1 = pd.concat([jan, feb, mar], ignore_index=True)  # the colleague's stack
answer_3 = [stack_q1.shape, int(stack_q1["customer_id"].isna().sum())]

In [ ]:
_ = check("Exercise 3", "answer_3", [(718, 6), 1], traps={((718, 7), 218): "The stack has a seventh column. Print `stack_q1.columns` and compare the spellings."})

<details>
<summary>Hint</summary>

Print `stack_q1.columns` and `stack_q1.isna().sum()`. Which column is nearly empty, and which new column appeared?

</details>

<details>
<summary>Solution</summary>

```python
jan = pd.read_csv("monthly/orders_2026_01.csv")
feb = pd.read_csv("monthly/orders_2026_02.csv").rename(columns={"customer_id": "customer"})
feb = feb.rename(columns={"customer": "customer_id"})
mar = pd.read_csv("monthly/orders_2026_03.csv")
stack_q1 = pd.concat([jan, feb, mar], ignore_index=True)
answer_3 = [stack_q1.shape, int(stack_q1["customer_id"].isna().sum())]
```

The stack has 718 orders and 6 columns, and one missing `customer_id`: the walk-in order of February. With the other spelling, `concat` kept both columns, so the stack had 7 columns and 218 missing ids, all of February, because the other files do not have a `customer` column; the check cell reports the shape.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")